# Experiment 2 · milton fold (batch 1) · Colab GPU training

Based on `docs/PLAN.md` v12 §7 and `docs/COLAB_RUN_GUIDE.md`. **This notebook runs only the milton fold**; FOLD is preset; do not change it.

**Order**: (1) settings -> (2) mount Drive -> (3) prepare code and data -> (4) check GPU (confirm A100) -> (5) smoke test -> (6) start training (evaluation runs automatically afterwards) -> (7) watch progress (about 2 h) -> disconnect the session when done

**Resume after disconnect**: after reconnecting run (1) (2) (3) (4) (6) in order ((6) detects the checkpoint and adds `--resume`; it refuses to start, with the reason, if the log was updated within 5 minutes or already records that training finished), then use (7) to watch progress.

**Run (8) manually only when needed**: (6) already passes `--eval-after`, so evaluation normally runs automatically after training; run (8) only if (6) reports "evaluation not finished yet".

## (1) Settings (FOLD is preset, do not change)

In [ ]:
FOLD = 'milton'   # batch 1; this notebook is fixed to the milton fold, do not change

DRIVE_ROOT = '/content/drive/MyDrive/Flood2'
PROJ       = '/content/Flood_2.0'                  # local working directory (code and unpacked data)
DATA_DIR   = f'{PROJ}/data/modelB_v1'
OUT_ROOT   = f'{DRIVE_ROOT}/models'                # training output goes to Drive, so runs can resume after a disconnect
FOLD_DIR   = f'{OUT_ROOT}/exp2/fold_{FOLD}'
assert FOLD in ['irma', 'matthew', 'ian', 'milton', 'dorian', 'beryl'], FOLD
print('FOLD =', FOLD, '\noutput directory =', FOLD_DIR)

## (2) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for f in ['modelB_v1.tar', 'modelB_v1.tar.md5', 'modelB_v1_manifest.csv', 'code/38_exp2_train.py']:
    p = f'{DRIVE_ROOT}/{f}'; print(('OK  ' if os.path.exists(p) else 'MISSING ') + p)

## (3) Prepare code and data (copy -> unpack -> md5 check; skipped automatically if already done in this session)

In [ ]:
import os, shutil, subprocess, time
os.makedirs(f'{PROJ}/code', exist_ok=True); os.makedirs(f'{PROJ}/data', exist_ok=True)
for f in os.listdir(f'{DRIVE_ROOT}/code'):
    shutil.copy2(f'{DRIVE_ROOT}/code/{f}', f'{PROJ}/code/{f}')
print('code copied:', sorted(os.listdir(f'{PROJ}/code')))

ok_flag = f'{DATA_DIR}/.verified'
if os.path.exists(ok_flag):
    print('data already unpacked and verified, skipping')
else:
    t0 = time.time()
    shutil.copy2(f'{DRIVE_ROOT}/modelB_v1.tar', '/content/modelB_v1.tar')
    print('tar copied in %.0f s' % (time.time() - t0))
    tar_md5 = subprocess.run(['md5sum', '/content/modelB_v1.tar'], capture_output=True, text=True).stdout.split()[0]
    expect = open(f'{DRIVE_ROOT}/modelB_v1.tar.md5').read().split()[0]
    assert tar_md5 == expect, ('tar md5 mismatch', tar_md5, expect)
    print('tar md5 matches', tar_md5)
    subprocess.run(['tar', '-xf', '/content/modelB_v1.tar', '-C', f'{PROJ}/data'], check=True)
    os.remove('/content/modelB_v1.tar')
    r = subprocess.run(['python3', f'{PROJ}/code/41_pack_modelB_data.py', 'verify', '--data-dir', DATA_DIR,
                        '--manifest', f'{DRIVE_ROOT}/modelB_v1_manifest.csv'], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    assert r.returncode == 0, 'per-file md5 check failed, stopping'
    open(ok_flag, 'w').write('ok\n')
    print('unpack + verification done, %.0f s in total' % (time.time() - t0))

## (4) Check GPU and torch version

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, platform
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| python', platform.python_version())
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU: change the runtime type to GPU and rerun from (1)')

## (5) Smoke test (small data: 3 train runs / 1 val run / 2 epochs; output in /content/smoke, nothing written to Drive)

In [ ]:
import shutil; shutil.rmtree('/content/smoke', ignore_errors=True)
!cd {PROJ} && python3 -u code/38_exp2_train.py --fold {FOLD} --data-dir {DATA_DIR} --out-root /content/smoke --smoke 2>&1 | tail -n 25
print('---- resume check: raise the limit to 3 epochs and --resume ----')
!cd {PROJ} && python3 -u code/38_exp2_train.py --fold {FOLD} --data-dir {DATA_DIR} --out-root /content/smoke --smoke --epochs 3 --resume 2>&1 | grep -E "resume|epoch 3 / |training finished|Error|Traceback"
!cd {PROJ} && python3 -u code/39_exp2_eval.py --fold {FOLD} --data-dir {DATA_DIR} --out-root /content/smoke --smoke 2>&1 | tail -n 4

Expected: the first command runs 2 epochs and saves a checkpoint; the second prints "resume: starting from epoch 3" and finishes epoch 3; the third prints single-step / rollout RMSE for 1 run. If there is **no Traceback and the loss is finite**, continue (smoke-test numbers are not conclusions).

## (6) Start / resume training for this fold (background; auto --resume if a checkpoint exists; automatic evaluation after training; guarded against double start)

In [ ]:
import os, subprocess, shlex, time
os.makedirs(FOLD_DIR, exist_ok=True)
LOG = f'{FOLD_DIR}/train_stdout.log'
EVAL_DONE = os.path.exists(f'{FOLD_DIR}/eval_test/eval_summary.json')
running = subprocess.run(['pgrep', '-f', f'[3]8_exp2_train.py --fold {FOLD} '], capture_output=True, text=True).stdout.split()
reason = None
if running:
    reason = f'Training for this fold is already running in this session (PID {running}); not starting again. Use (7) to watch progress.'
elif os.path.exists(LOG):
    age = time.time() - os.path.getmtime(LOG)
    txt = open(LOG, errors='replace').read()
    if 'training finished' in txt or 'finished previously' in txt:
        reason = ('The log already records that training finished; not training again. '
                  + ('Evaluation is also done (eval_test/eval_summary.json exists); this fold is complete, you can disconnect.' if EVAL_DONE
                     else 'But evaluation not finished yet: please run (8).'))
    elif age < 300:
        reason = (f'The log was updated {age:.0f} s ago (less than 5 minutes); another session may be running the {FOLD} fold. '
                  'Check with colab_exp2_status.ipynb first; once sure no other session is running, wait until the log has not been updated for over 5 minutes, then rerun this cell (it resumes automatically).')
if reason:
    print('!! Refusing to start:', reason)
else:
    resume = os.path.exists(f'{FOLD_DIR}/checkpoint_last.pt')
    cmd = (f'cd {shlex.quote(PROJ)} && nohup python3 -u code/38_exp2_train.py --fold {FOLD} '
           f'--data-dir {shlex.quote(DATA_DIR)} --out-root {shlex.quote(OUT_ROOT)} --eval-after ' + ('--resume ' if resume else '') +
           f'>> {shlex.quote(LOG)} 2>&1 &')
    os.system(cmd)
    print(('Resuming' if resume else 'Fresh start') + ' (the 10 test runs are evaluated automatically after training):', cmd)
    print('Run (7) in a few tens of seconds to watch progress; keep this page open and do not let the computer sleep')

## (7) Watch progress (refresh every 60 s, 120 times, about 2 h; you can interrupt this cell at any time without affecting training)

In [ ]:
%run {PROJ}/code/40_watch_colab.py --fold {FOLD} --out-root {OUT_ROOT} --interval 60 --count 120

In [ ]:
# show once + whether the process is still alive
!pgrep -fl "[3]8_exp2_train.py --fold {FOLD} " || echo "not running (finished / early-stopped / disconnected)"
%run {PROJ}/code/40_watch_colab.py --fold {FOLD} --out-root {OUT_ROOT}

## (8) Manual evaluation (run only if (6) reports "evaluation not finished yet"): 10 test runs of the held-out event, single-step + rollout + persistence baseline, all 168 time steps

In [ ]:
!pgrep -fl "[3]8_exp2_train.py --fold {FOLD} " && echo "!! training still running, evaluate after it finishes" || (cd {PROJ} && python3 -u code/39_exp2_eval.py --fold {FOLD} --data-dir {DATA_DIR} --out-root {OUT_ROOT})
# to evaluate all 81 runs of the held-out event, add --all-runs

## (9) Record GPU model and measured time (copy results into docs/PROGRESS.md)

In [ ]:
import json, csv, re
info = json.load(open(f'{FOLD_DIR}/fold_info.json'))
for s in info['sessions']:
    print('session', s['started_at'], '| GPU', s.get('gpu', s['device']), '| torch', s['torch'], '| from epoch', s['start_epoch'], '(resumed)' if s['resume'] else '')
rows = list(csv.DictReader(open(f'{FOLD_DIR}/train_log.csv')))
if rows:
    secs = [float(r['epoch_sec']) for r in rows]
    best = min(rows, key=lambda r: float(r['val_loss']))
    print('%d epochs done, %.2f h total, %.0f s/epoch on average; best epoch %s val %s' % (len(rows), sum(secs) / 3600, sum(secs) / len(secs), best['epoch'], best['val_loss']))
sps = [float(m.group(1)) for m in re.finditer(r'\| ([\d.]+) s/step', open(f'{FOLD_DIR}/train_stdout.log').read())]
if sps: print('s/step: median %.3f (%d progress lines)' % (sorted(sps)[len(sps) // 2], len(sps)))